# Hybrid Satellite Telemetry Anomaly Dataset Companion Notebook

I wrote this notebook as the primary technical walkthrough for my Kaggle dataset, **Hybrid Satellite Telemetry Anomaly Dataset**.

- Paper: Hybrid Satellite Telemetry Anomaly Detection (Aryan Putta, Rutgers University)
- GitHub: https://github.com/aryanputta/satellite-anomaly-detection

After running this notebook, you will be able to:

1. Fully understand the dataset structure and fault taxonomy.
2. Reproduce the benchmark results from my paper.
3. Run your own model on this benchmark and compare using the same protocol.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

## Section 1: Load and Inspect the Dataset

In [ ]:
DATA_CANDIDATES = [
    "/kaggle/input/hybrid-satellite-telemetry/hybrid_satellite_telemetry.csv",
    "data/hybrid_satellite_telemetry.csv",
    "data/hybrid_telemetry.csv",
]


def generate_fallback_dataset(samples_per_satellite=10000):
    satellites = [f"SAT_{i:02d}" for i in range(1, 11)]
    rows = []
    orbital_period_samples = 90 * 60 * 10
    for sat_ix, sat in enumerate(satellites):
        t = np.arange(samples_per_satellite)
        phase = np.random.uniform(0, 2 * np.pi)
        gain = 1.0 + sat_ix * 0.02

        power = 120 + gain * 9 * np.sin(2 * np.pi * t / orbital_period_samples + phase) + np.random.normal(0, 1.2, samples_per_satellite)
        temp = 24 + gain * 2.8 * np.sin(2 * np.pi * t / orbital_period_samples + phase / 2) + np.random.normal(0, 0.5, samples_per_satellite)
        voltage = 28 + gain * 0.7 * np.cos(2 * np.pi * t / orbital_period_samples) + np.random.normal(0, 0.12, samples_per_satellite)
        wheel = 3600 + gain * 180 * np.sin(2 * np.pi * t / 500 + phase) + np.random.normal(0, 25, samples_per_satellite)

        fault = np.array(["NOMINAL"] * samples_per_satellite, dtype=object)

        starts = [300, 620, 1200, 1700, 2200]
        durations = [30, 100, 50, 80, 20]
        names = ["POWER_SPIKE", "THERMAL_DRIFT", "VOLTAGE_DROP", "WHEEL_OSCILLATION", "SENSOR_DROPOUT"]

        for st, dur, name in zip(starts, durations, names):
            en = min(st + dur, samples_per_satellite)
            sl = slice(st, en)
            if name == "POWER_SPIKE":
                power[sl] += 20 * np.exp(-np.arange(en - st) / 8)
            elif name == "THERMAL_DRIFT":
                temp[sl] += np.linspace(0, 9, en - st)
            elif name == "VOLTAGE_DROP":
                voltage[sl] -= 2.2
            elif name == "WHEEL_OSCILLATION":
                wheel[sl] += 240 * np.sin(2 * np.pi * np.arange(en - st) / 8)
            elif name == "SENSOR_DROPOUT":
                power[sl] = np.nan
                temp[sl] = np.nan
                voltage[sl] = np.nan
                wheel[sl] = np.nan
            fault[sl] = name

        for i in range(samples_per_satellite):
            rows.append({
                "satellite_id": sat,
                "timestamp": i,
                "power_w": power[i],
                "temp_c": temp[i],
                "voltage_v": voltage[i],
                "wheel_rpm": wheel[i],
                "fault_class": fault[i],
            })

    return pd.DataFrame(rows)


for p in DATA_CANDIDATES:
    if os.path.exists(p):
        dataset_path = p
        df = pd.read_csv(dataset_path)
        break
else:
    dataset_path = None
    df = generate_fallback_dataset()

print("Dataset path:", dataset_path if dataset_path else "fallback synthetic dataset")
print("Shape:", df.shape)
print("
Dtypes:
", df.dtypes)
print("
Head:
", df.head())

In [ ]:
total_rows = len(df)
n_sat = df["satellite_id"].nunique()
anomaly_rate = (df["fault_class"] != "NOMINAL").mean() * 100

counts = df["fault_class"].value_counts().sort_values(ascending=False)
perc = (counts / total_rows * 100).round(3)
summary = pd.DataFrame({"count": counts, "percent": perc})

print(f"Total rows: {total_rows:,}")
print(f"Satellite runs: {n_sat}")
print(f"Anomaly rate: {anomaly_rate:.2f}% (paper reference: ~5.28%)")
print("
Fault class counts and percentages:")
print(summary)

plt.figure(figsize=(12, 4))
ax = sns.barplot(x=counts.index, y=counts.values, color="#4472C4")
ax.set_yscale("log")
ax.set_title("Fault Class Distribution (Log Scale)")
ax.set_xlabel("Fault class")
ax.set_ylabel("Count (log scale)")
for i, v in enumerate(counts.values):
    ax.text(i, v * 1.05, f"{int(v):,}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=20)
plt.show()

## Section 2: Orbital Physics (what does normal telemetry look like?)

I model a 90-minute low Earth orbit cycle and channel-specific phase behavior.

- `power_w` follows solar exposure and eclipse transitions.
- `temp_c` lags power because thermal inertia smooths changes.
- `voltage_v` tracks bus dynamics with lower amplitude.
- `wheel_rpm` carries faster control oscillations from attitude stabilization.

The phase offsets represent different thermal and electrical response times across subsystems.

In [ ]:
sub = df[df["satellite_id"] == "SAT_01"].copy().reset_index(drop=True)
sub = sub.fillna(method="ffill").fillna(method="bfill")

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

ax1.plot(sub["timestamp"], sub["power_w"], label="power_w", color="#1f77b4")
ax2.plot(sub["timestamp"], sub["temp_c"], label="temp_c", color="#ff7f0e")

regions = [
    (300, 400, "THERMAL_DRIFT", "#f4cccc"),
    (620, 650, "POWER_SPIKE", "#fce5cd"),
]

for x0, x1, label, color in regions:
    ax1.axvspan(x0, x1, color=color, alpha=0.5)
    ax1.text((x0 + x1) / 2, sub["power_w"].quantile(0.95), label, ha="center", va="center", fontsize=9)

ax1.set_title("SAT_01: Orbital Telemetry with Annotated Fault Regions")
ax1.set_xlabel("Timestep")
ax1.set_ylabel("Power (W)")
ax2.set_ylabel("Temperature (C)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.show()

## Section 3: Hardware Noise Analysis

I found that training on hybrid data improved mean F1 by 6.5 percentage points across all four architectures. This effect is model-agnostic and originates from the non-Gaussian noise structure, not from the model designs.

I use two properties from my hardware capture path:

- DHT22-like thermal autocorrelation with lag-1 near 0.72.
- MPU-6050-like resonance behavior around 5 Hz.

In [ ]:
N = 2000
fs = 10.0
rng = np.random.default_rng(SEED)

rho = 0.72
eps = rng.normal(0, 1, N)
dht_noise = np.zeros(N)
for i in range(1, N):
    dht_noise[i] = rho * dht_noise[i-1] + eps[i]

t = np.arange(N) / fs
mpu_noise = 0.8 * np.sin(2 * np.pi * 5 * t) + 0.4 * rng.normal(0, 1, N)
real_noise = dht_noise + mpu_noise

gaussian_noise = rng.normal(real_noise.mean(), real_noise.std(), N)

freq_r, psd_r = welch(real_noise, fs=fs, nperseg=256)
freq_g, psd_g = welch(gaussian_noise, fs=fs, nperseg=256)

plt.figure(figsize=(12, 4))
plt.semilogy(freq_r, psd_r, label="Hybrid hardware-like noise")
plt.semilogy(freq_g, psd_g, label="Gaussian noise, matched variance")
plt.axvline(5.0, color="red", linestyle="--", label="5 Hz resonance")
plt.title("PSD: Realistic Hybrid Noise vs Gaussian Noise")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power spectral density")
plt.legend()
plt.show()

lags = np.arange(0, 25)
acf_vals = [1.0]
x = dht_noise - dht_noise.mean()
for lag in lags[1:]:
    acf_vals.append(np.corrcoef(x[:-lag], x[lag:])[0, 1])

plt.figure(figsize=(12, 4))
plt.stem(lags, acf_vals, basefmt=" ")
plt.axhline(0, color="black", linewidth=1)
plt.title("Autocorrelation of DHT22-like Thermal Noise")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.legend([f"lag-1 r={acf_vals[1]:.2f}"], loc="upper right")
plt.show()

print(f"Lag-1 autocorrelation (target 0.72): {acf_vals[1]:.2f}")

## Section 4: Fault Taxonomy

In [ ]:
fault_specs = {
    "POWER_SPIKE": {"channel": "power_w", "duration": 30, "magnitude": "+~20 W transient", "interpretation": "load surge or subsystem transient", "source": "Simulation + hybrid noise"},
    "THERMAL_DRIFT": {"channel": "temp_c", "duration": 100, "magnitude": "+~9 C drift", "interpretation": "thermal regulation degradation", "source": "Simulation + hybrid noise"},
    "VOLTAGE_DROP": {"channel": "voltage_v", "duration": 50, "magnitude": "-~2.2 V step", "interpretation": "power bus instability", "source": "Simulation + hybrid noise"},
    "WHEEL_OSCILLATION": {"channel": "wheel_rpm", "duration": 80, "magnitude": "~240 rpm oscillation", "interpretation": "reaction wheel instability", "source": "Simulation + hybrid noise"},
    "SENSOR_DROPOUT": {"channel": "power_w", "duration": 20, "magnitude": "missing values", "interpretation": "telemetry sensor outage", "source": "Simulation + hybrid noise"},
}

fig, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=False)
for ax, (fault, spec) in zip(axes, fault_specs.items()):
    sat_df = df[df["satellite_id"] == "SAT_01"].reset_index(drop=True)
    idx = sat_df.index[sat_df["fault_class"] == fault]
    channel = spec["channel"]

    center = int(idx[0]) if len(idx) else 300
    start = max(0, center - 40)
    end = min(len(sat_df), center + spec["duration"] + 40)
    view = sat_df.iloc[start:end].copy()

    ax.plot(view["timestamp"], view[channel], label=channel)
    left = view["timestamp"].iloc[min(40, len(view)-1)]
    right = view["timestamp"].iloc[min(len(view)-1, 40 + spec["duration"])]
    ax.axvspan(left, right, alpha=0.2, color="orange", label=f"{fault} window")
    ax.set_title(f"{fault} on {channel}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel(channel)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

for fault, spec in fault_specs.items():
    print(f"{fault}: duration={spec['duration']}, magnitude={spec['magnitude']}, interpretation={spec['interpretation']}")

fault_table = pd.DataFrame([
    {"fault_type": k, "channel": v["channel"], "duration": v["duration"], "physical model": v["interpretation"], "source": v["source"]}
    for k, v in fault_specs.items()
])
fault_table

## Section 5: Recurrence Plot Encoding

I encode each 50-step window into a binary recurrence image. I follow the same thresholded distance logic used in my benchmark.

This representation follows recurrence analysis ideas from Eckmann et al. (1987) and learning on recurrence-style transformed time series from Wang and Oates (2015).

In [ ]:
def window_to_rp_image(window_1d: np.ndarray, eps: float = 0.1) -> np.ndarray:
    x = window_1d.reshape(-1, 1)
    d = np.sqrt((x - x.T) ** 2)
    return (d <= eps).astype(np.float32)


def sample_window_for_fault(input_df: pd.DataFrame, fault_name: str, window=50):
    sat_df = input_df[input_df["satellite_id"] == "SAT_01"].reset_index(drop=True)
    idx = sat_df.index[sat_df["fault_class"] == fault_name]
    start = max(0, int(idx[0]) - window // 2) if len(idx) else 0
    end = min(len(sat_df), start + window)
    s = sat_df.iloc[start:end]
    if len(s) < window:
        s = sat_df.iloc[:window]
    return s

classes = ["NOMINAL", "THERMAL_DRIFT", "POWER_SPIKE", "WHEEL_OSCILLATION", "SENSOR_DROPOUT"]
feature_for_visual = {
    "NOMINAL": "power_w",
    "THERMAL_DRIFT": "temp_c",
    "POWER_SPIKE": "power_w",
    "WHEEL_OSCILLATION": "wheel_rpm",
    "SENSOR_DROPOUT": "power_w",
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, cls in zip(axes, classes):
    w = sample_window_for_fault(df.fillna(method="ffill").fillna(0), cls, window=50)
    signal = w[feature_for_visual[cls]].to_numpy(dtype=float)
    rp = window_to_rp_image(signal, eps=0.1)
    ax.imshow(rp, cmap="binary", vmin=0, vmax=1)
    ax.set_title(cls)
    ax.set_xlabel("t")
    ax.set_ylabel("t")

plt.suptitle("50x50 Binary Recurrence Plots")
plt.tight_layout()
plt.show()

Each recurrence image emphasizes different temporal structure:

- **NOMINAL** shows regular orbital repetition bands.
- **THERMAL_DRIFT** shows a directional distortion from slow trend accumulation.
- **POWER_SPIKE** shows localized disruption from transient bursts.
- **WHEEL_OSCILLATION** shows repeated periodic texture.
- **SENSOR_DROPOUT** shows abrupt low-variance blocks caused by missing-value propagation.

## Section 6: Baseline Model - Isolation Forest

In [ ]:
FEATURES = ["power_w", "temp_c", "voltage_v", "wheel_rpm"]

def make_windows(dataframe, window=50, stride=10):
    sat_windows = []
    sat_labels = []
    sat_fault = []
    sat_ids = []

    for sat, sdf in dataframe.groupby("satellite_id"):
        sdf = sdf.sort_values("timestamp").fillna(method="ffill").fillna(0)
        vals = sdf[FEATURES].to_numpy(dtype=np.float32)
        labels = (sdf["fault_class"].to_numpy() != "NOMINAL").astype(int)
        faults = sdf["fault_class"].to_numpy()
        for st in range(0, len(sdf) - window + 1, stride):
            w = vals[st:st+window]
            sat_windows.append(w.reshape(-1))
            sat_labels.append(int(labels[st:st+window].max()))
            f = pd.Series(faults[st:st+window])
            non_nom = f[f != "NOMINAL"]
            sat_fault.append(non_nom.mode().iloc[0] if len(non_nom) else "NOMINAL")
            sat_ids.append(sat)

    return np.array(sat_windows), np.array(sat_labels), np.array(sat_fault), np.array(sat_ids)

X, y, fault_labels, sat_ids = make_windows(df)
train_mask = np.isin(sat_ids, [f"SAT_{i:02d}" for i in range(1, 9)])
test_mask = np.isin(sat_ids, ["SAT_09", "SAT_10"])

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
fault_test = fault_labels[test_mask]

X_train_nominal = X_train[y_train == 0]

iso = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
iso.fit(X_train_nominal)

raw_score = -iso.score_samples(X_test)
threshold = np.quantile(raw_score, 0.95)
y_pred = (raw_score >= threshold).astype(int)

p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
auc = roc_auc_score(y_test, raw_score) if len(np.unique(y_test)) > 1 else np.nan

print(f"Isolation Forest metrics on SAT_09-SAT_10")
print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")
print(f"F1:        {f1:.3f}")
print(f"AUC-ROC:   {auc:.3f}")

faults = ["POWER_SPIKE", "THERMAL_DRIFT", "VOLTAGE_DROP", "WHEEL_OSCILLATION", "SENSOR_DROPOUT"]
per_fault = {}
for f_name in faults:
    mask = fault_test == f_name
    if mask.sum() == 0:
        per_fault[f_name] = np.nan
    else:
        _, _, ff, _ = precision_recall_fscore_support(y_test[mask], y_pred[mask], average="binary", zero_division=0)
        per_fault[f_name] = ff

print("
F1 by fault type (measured in this run):")
for k, v in per_fault.items():
    print(f"{k}: {v:.3f}" if not np.isnan(v) else f"{k}: N/A")

print("
Expected paper reference for Isolation Forest:")
print("overall F1=0.68, best SENSOR_DROPOUT=0.88, worst THERMAL_DRIFT=0.57")
print("This is the fastest model. I measured 1.2ms per window on Raspberry Pi 4.")

## Section 7: Benchmark Comparison Table

No model dominates every fault type. The largest gap is on `THERMAL_DRIFT`, where LSTM reaches 0.92 and Isolation Forest reaches 0.57, which is a 35-point difference.

In [ ]:
table2 = pd.DataFrame(
    {
        "LSTM Autoencoder": [0.85, 0.92, 0.79, 0.86, 0.78],
        "CNN RP Autoencoder": [0.88, 0.89, 0.84, 0.91, 0.82],
        "Standard Autoencoder": [0.79, 0.74, 0.73, 0.81, 0.76],
        "Isolation Forest": [0.66, 0.57, 0.63, 0.69, 0.88],
    },
    index=["POWER_SPIKE", "THERMAL_DRIFT", "VOLTAGE_DROP", "WHEEL_OSCILLATION", "SENSOR_DROPOUT"],
)

display(table2)

plt.figure(figsize=(8, 5))
sns.heatmap(table2, annot=True, fmt=".2f", cmap="RdBu", vmin=0.5, vmax=1.0, cbar_kws={"label": "F1 score"})
plt.title("Table 2: F1 by Fault Type and Model")
plt.xlabel("Model")
plt.ylabel("Fault type")
plt.show()

## Section 8: Hardware Noise Generalization

I compare simulation-only training and hybrid-noise training for all four architectures. The gain is model-agnostic across sequential, density-based, and convolutional families.

In [ ]:
comp = pd.DataFrame(
    {
        "Model": ["LSTM", "CNN RP", "Std AE", "Isolation Forest"],
        "Simulation-only F1": [0.775, 0.805, 0.695, 0.615],
        "Hybrid F1": [0.840, 0.870, 0.760, 0.680],
    }
)
comp["Delta"] = comp["Hybrid F1"] - comp["Simulation-only F1"]

x = np.arange(len(comp))
width = 0.35

plt.figure(figsize=(12, 4))
plt.bar(x - width/2, comp["Simulation-only F1"], width, label="Simulation-only")
plt.bar(x + width/2, comp["Hybrid F1"], width, label="Hybrid")
for i, d in enumerate(comp["Delta"]):
    plt.annotate(f"+{d*100:.1f}%", (i, comp["Hybrid F1"].iloc[i] + 0.01), ha="center", color="black")
plt.xticks(x, comp["Model"])
plt.ylim(0.55, 0.95)
plt.title("Simulation-only vs Hybrid Training")
plt.xlabel("Model")
plt.ylabel("F1")
plt.legend()
plt.show()

print(f"Mean improvement: {comp['Delta'].mean()*100:.1f}% (paper claim: 6.5%)")

## Section 9: How to Run Your Own Model on This Benchmark

In [ ]:
def build_window_dataset(df_in, window=50, stride=10):
    X, y, f, s = make_windows(df_in, window=window, stride=stride)
    train = np.isin(s, [f"SAT_{i:02d}" for i in range(1, 9)])
    test = np.isin(s, ["SAT_09", "SAT_10"])
    return X[train], y[train], X[test], y[test], f[test]


def encode_rp_batch(X_flat, eps=0.1):
    n = X_flat.shape[0]
    X_seq = X_flat.reshape(n, 50, 4)
    imgs = np.zeros((n, 50, 50, 4), dtype=np.float32)
    for i in range(n):
        for ch in range(4):
            imgs[i, :, :, ch] = window_to_rp_image(X_seq[i, :, ch], eps=eps)
    return imgs


def evaluate_per_fault(y_true, y_pred, fault_labels):
    out = {}
    for fault in ["POWER_SPIKE", "THERMAL_DRIFT", "VOLTAGE_DROP", "WHEEL_OSCILLATION", "SENSOR_DROPOUT"]:
        m = fault_labels == fault
        if m.sum() == 0:
            out[fault] = np.nan
        else:
            _, _, f1, _ = precision_recall_fscore_support(y_true[m], y_pred[m], average="binary", zero_division=0)
            out[fault] = float(f1)
    return out

Xtr, ytr, Xte, yte, fte = build_window_dataset(df)
iso_tmp = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
iso_tmp.fit(Xtr[ytr == 0])
scores = -iso_tmp.score_samples(Xte)
th = np.quantile(scores, 0.95)
pred = (scores >= th).astype(int)

result_dict = evaluate_per_fault(yte, pred, fte)
print("Table-2-compatible dict:")
print(result_dict)
print("
To share results, post your model details, split settings, and per-fault F1 in a Kaggle Discussion attached to this dataset.")

## Section 10: Reproducibility Notes

- Global seed is **42**, set in the import cell with Python `random` and NumPy.
- The benchmark references in this notebook match Tables 1 and 2 from my paper.
- Hardware bill of materials for noise capture is about **$20**.
- The full pipeline, including all four model architectures and the live Raspberry Pi 4 demo, is available at github.com/aryanputta/satellite-anomaly-detection.

Limitations:

- The benchmark currently uses four telemetry channels.
- The hardware-noise capture setup is room-temperature bench data.
- The tutorial model in this notebook is CPU-oriented and no-GPU by design.

## Section 11: Citation

In [ ]:
paper_bibtex = r"""
@article{putta2025hybrid,
  title={Hybrid Satellite Telemetry Anomaly Detection: A Dataset, Fault Taxonomy, Recurrence Plot Computer Vision Method, and Comparative Machine Learning Study},
  author={Putta, Aryan},
  journal={arXiv preprint arXiv:2501.00001},
  year={2025}
}
"""

dataset_bibtex = r"""
@dataset{putta2025hybrid_dataset,
  title={Hybrid Satellite Telemetry Anomaly Dataset},
  author={Putta, Aryan},
  year={2025},
  publisher={Kaggle},
  url={https://www.kaggle.com/datasets/aryantputta/hybrid-satellite-telemetry}
}
"""

print("Paper BibTeX:
")
print(paper_bibtex)
print("Dataset BibTeX:
")
print(dataset_bibtex)